# Sparse-Filtered Row-Wise LGBM Controlled Tuning

Tune `LGBMRegressor` one parameter at a time using the best feature setup from previous experiments: sparse threshold `0.9875` plus row-wise aggregate features.

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd

In [2]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split

from src.loader import Loader
from src.preprocessing import FeaturePreprocessor

In [3]:
SEED = 42
TEST_SIZE = 0.33
CV = 5
SPARSITY_THRESHOLD = 0.9875

In [4]:
loader = Loader()
df = loader.load("../data/processed_data.csv")
df.shape

(4459, 4732)

In [5]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

In [6]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

preprocessor = FeaturePreprocessor(
    zero_share_threshold=SPARSITY_THRESHOLD,
    add_rowwise=True,
)
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

pd.DataFrame(
    {
        "metric": ["sparsity_threshold", "base_features_kept", "added_rowwise_features", "final_features"],
        "value": [
            SPARSITY_THRESHOLD,
            len(preprocessor.columns_to_keep_),
            X_train.shape[1] - len(preprocessor.columns_to_keep_),
            X_train.shape[1],
        ],
    }
)

,metric,value
0,sparsity_threshold,0.9875
1,base_features_kept,2482.0000
2,added_rowwise_features,8.0000
3,final_features,2490.0000


The notebook uses the strongest feature setup from the previous experiments before tuning: sparse filtering from notebook `05`, then row-wise aggregate features from notebook `06`. The tuning strategy remains controlled: sweep one parameter, update `current_params` to the best candidate, then move to the next parameter.

In [7]:
current_params = {
    "learning_rate": 0.03,
    "n_estimators": 500,
    "num_leaves": 31,
    "max_depth": 8,
    "min_child_samples": 20,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.05,
    "reg_lambda": 0.05,
    "min_split_gain": 0.0,
}

current_params

{'learning_rate': 0.03,
 'n_estimators': 500,
 'num_leaves': 31,
 'max_depth': 8,
 'min_child_samples': 20,
 'subsample': 0.8,
 'subsample_freq': 1,
 'colsample_bytree': 0.8,
 'reg_alpha': 0.05,
 'reg_lambda': 0.05,
 'min_split_gain': 0.0}

In [8]:
sweeps = {
    "num_leaves": [15, 31, 63, 95],
    "max_depth": [6, 8, 10, 12],
    "min_child_samples": [5, 10, 20, 50],
    "learning_rate": [0.01, 0.02, 0.03, 0.05],
    "n_estimators": [300, 500, 700, 1000],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_alpha": [0.0, 0.05, 0.1, 0.5],
    "reg_lambda": [0.0, 0.05, 0.1, 0.5],
}

list(sweeps.keys())

['num_leaves',
 'max_depth',
 'min_child_samples',
 'learning_rate',
 'n_estimators',
 'subsample',
 'colsample_bytree',
 'reg_alpha',
 'reg_lambda']

In [9]:
def evaluate_params(params: dict) -> tuple[float, float]:
    model = LGBMRegressor(
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
        **params,
    )
    # RMSE in log-space is equivalent to RMSLE for the log-target setup.
    scores = -cross_val_score(
        estimator=model,
        X=X_train,
        y=y_train_log,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=1,
    )
    return scores.mean(), scores.std()


def run_single_parameter_sweep(parameter_name: str, candidate_values: list, base_params: dict) -> pd.DataFrame:
    sweep_rows = []

    for candidate_value in candidate_values:
        trial_params = base_params.copy()
        trial_params[parameter_name] = candidate_value
        cv_mean, cv_std = evaluate_params(trial_params)
        sweep_rows.append(
            {
                "sweep_parameter": parameter_name,
                "candidate_value": candidate_value,
                "cv_rmsle_mean": cv_mean,
                "cv_rmsle_std": cv_std,
                **trial_params,
            }
        )

    return pd.DataFrame(sweep_rows).sort_values(by="cv_rmsle_mean").reset_index(drop=True)

In [10]:
all_sweep_results = []
selected_steps = []

for parameter_name, candidate_values in sweeps.items():
    sweep_df = run_single_parameter_sweep(parameter_name, candidate_values, current_params)
    display(sweep_df.style.format({"cv_rmsle_mean": "{:,.4f}", "cv_rmsle_std": "{:,.4f}"}))
    all_sweep_results.append(sweep_df)

    best_row = sweep_df.iloc[0]
    current_params[parameter_name] = best_row["candidate_value"].item() if hasattr(best_row["candidate_value"], "item") else best_row["candidate_value"]
    selected_steps.append(
        {
            "parameter": parameter_name,
            "selected_value": current_params[parameter_name],
            "cv_rmsle_mean": float(best_row["cv_rmsle_mean"]),
            "cv_rmsle_std": float(best_row["cv_rmsle_std"]),
        }
    )

current_params

,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,min_split_gain
0,num_leaves,31,1.3757,0.0223,0.030000,500,31,8,20,0.800000,1,0.800000,0.050000,0.050000,0.000000
1,num_leaves,63,1.3769,0.0303,0.030000,500,63,8,20,0.800000,1,0.800000,0.050000,0.050000,0.000000
2,num_leaves,95,1.3769,0.0303,0.030000,500,95,8,20,0.800000,1,0.800000,0.050000,0.050000,0.000000
3,num_leaves,15,1.3813,0.0321,0.030000,500,15,8,20,0.800000,1,0.800000,0.050000,0.050000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,min_split_gain
0,max_depth,6,1.3739,0.0301,0.030000,500,31,6,20,0.800000,1,0.800000,0.050000,0.050000,0.000000
1,max_depth,8,1.3757,0.0223,0.030000,500,31,8,20,0.800000,1,0.800000,0.050000,0.050000,0.000000
2,max_depth,10,1.3819,0.0213,0.030000,500,31,10,20,0.800000,1,0.800000,0.050000,0.050000,0.000000
3,max_depth,12,1.3886,0.0234,0.030000,500,31,12,20,0.800000,1,0.800000,0.050000,0.050000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,min_split_gain
0,min_child_samples,10,1.3692,0.0354,0.030000,500,31,6,10,0.800000,1,0.800000,0.050000,0.050000,0.000000
1,min_child_samples,5,1.3700,0.0350,0.030000,500,31,6,5,0.800000,1,0.800000,0.050000,0.050000,0.000000
2,min_child_samples,20,1.3739,0.0301,0.030000,500,31,6,20,0.800000,1,0.800000,0.050000,0.050000,0.000000
3,min_child_samples,50,1.3846,0.0323,0.030000,500,31,6,50,0.800000,1,0.800000,0.050000,0.050000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,min_split_gain
0,learning_rate,0.010000,1.3599,0.0345,0.010000,500,31,6,10,0.800000,1,0.800000,0.050000,0.050000,0.000000
1,learning_rate,0.020000,1.3624,0.0318,0.020000,500,31,6,10,0.800000,1,0.800000,0.050000,0.050000,0.000000
2,learning_rate,0.030000,1.3692,0.0354,0.030000,500,31,6,10,0.800000,1,0.800000,0.050000,0.050000,0.000000
3,learning_rate,0.050000,1.3906,0.0233,0.050000,500,31,6,10,0.800000,1,0.800000,0.050000,0.050000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,min_split_gain
0,n_estimators,500,1.3599,0.0345,0.010000,500,31,6,10,0.800000,1,0.800000,0.050000,0.050000,0.000000
1,n_estimators,700,1.3599,0.0318,0.010000,700,31,6,10,0.800000,1,0.800000,0.050000,0.050000,0.000000
2,n_estimators,1000,1.3617,0.0295,0.010000,1000,31,6,10,0.800000,1,0.800000,0.050000,0.050000,0.000000
3,n_estimators,300,1.3643,0.0359,0.010000,300,31,6,10,0.800000,1,0.800000,0.050000,0.050000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,min_split_gain
0,subsample,0.800000,1.3599,0.0345,0.010000,500,31,6,10,0.800000,1,0.800000,0.050000,0.050000,0.000000
1,subsample,0.700000,1.3616,0.0363,0.010000,500,31,6,10,0.700000,1,0.800000,0.050000,0.050000,0.000000
2,subsample,0.900000,1.3629,0.0349,0.010000,500,31,6,10,0.900000,1,0.800000,0.050000,0.050000,0.000000
3,subsample,1.000000,1.3675,0.0363,0.010000,500,31,6,10,1.000000,1,0.800000,0.050000,0.050000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,min_split_gain
0,colsample_bytree,1.000000,1.3585,0.0366,0.010000,500,31,6,10,0.800000,1,1.000000,0.050000,0.050000,0.000000
1,colsample_bytree,0.800000,1.3599,0.0345,0.010000,500,31,6,10,0.800000,1,0.800000,0.050000,0.050000,0.000000
2,colsample_bytree,0.600000,1.3623,0.0374,0.010000,500,31,6,10,0.800000,1,0.600000,0.050000,0.050000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,min_split_gain
0,reg_alpha,0.500000,1.3573,0.0365,0.010000,500,31,6,10,0.800000,1,1.000000,0.500000,0.050000,0.000000
1,reg_alpha,0.050000,1.3585,0.0366,0.010000,500,31,6,10,0.800000,1,1.000000,0.050000,0.050000,0.000000
2,reg_alpha,0.000000,1.3588,0.0366,0.010000,500,31,6,10,0.800000,1,1.000000,0.000000,0.050000,0.000000
3,reg_alpha,0.100000,1.3599,0.0354,0.010000,500,31,6,10,0.800000,1,1.000000,0.100000,0.050000,0.000000


,sweep_parameter,candidate_value,cv_rmsle_mean,cv_rmsle_std,learning_rate,n_estimators,num_leaves,max_depth,min_child_samples,subsample,subsample_freq,colsample_bytree,reg_alpha,reg_lambda,min_split_gain
0,reg_lambda,0.000000,1.3571,0.0365,0.010000,500,31,6,10,0.800000,1,1.000000,0.500000,0.000000,0.000000
1,reg_lambda,0.500000,1.3573,0.0363,0.010000,500,31,6,10,0.800000,1,1.000000,0.500000,0.500000,0.000000
2,reg_lambda,0.050000,1.3573,0.0365,0.010000,500,31,6,10,0.800000,1,1.000000,0.500000,0.050000,0.000000
3,reg_lambda,0.100000,1.3573,0.0355,0.010000,500,31,6,10,0.800000,1,1.000000,0.500000,0.100000,0.000000


{'learning_rate': 0.01,
 'n_estimators': 500,
 'num_leaves': 31,
 'max_depth': 6,
 'min_child_samples': 10,
 'subsample': 0.8,
 'subsample_freq': 1,
 'colsample_bytree': 1.0,
 'reg_alpha': 0.5,
 'reg_lambda': 0.0,
 'min_split_gain': 0.0}

The loop above updates `current_params` after each sweep. This keeps the tuning process simple and auditable while allowing later sweeps to use the best value found in earlier sweeps.

In [11]:
sweep_results_df = pd.concat(all_sweep_results, ignore_index=True)
selected_steps_df = pd.DataFrame(selected_steps)
selected_steps_df

,parameter,selected_value,cv_rmsle_mean,cv_rmsle_std
0,num_leaves,31.00,1.375659,0.022256
1,max_depth,6.00,1.373929,0.030051
2,min_child_samples,10.00,1.369204,0.035416
3,learning_rate,0.01,1.359907,0.034481
4,n_estimators,500.00,1.359907,0.034481
5,subsample,0.80,1.359907,0.034481
6,colsample_bytree,1.00,1.358516,0.036624
7,reg_alpha,0.50,1.357312,0.036474
8,reg_lambda,0.00,1.357100,0.036529


In [12]:
best_model = LGBMRegressor(
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
    **current_params,
)

best_model.fit(X_train, y_train_log)

y_pred_log = best_model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_pred = np.clip(y_pred, 0, None)

In [13]:
metrics = pd.DataFrame(
    {
        "metric": ["rmsle", "rmse", "mae", "r2"],
        "value": [
            root_mean_squared_log_error(y_test_raw, y_pred),
            root_mean_squared_error(y_test_raw, y_pred),
            mean_absolute_error(y_test_raw, y_pred),
            r2_score(y_test_raw, y_pred),
        ],
    }
)

metrics.style.format({"value": "{:,.4f}"})

,metric,value
0,rmsle,1.3776
1,rmse,"7,030,480.0963"
2,mae,"3,925,190.7586"
3,r2,0.2256


In [14]:
summary = {
    "target_transform": "log1p",
    "primary_metric": "rmsle",
    "feature_setup": "sparse_threshold_0.9875_plus_rowwise",
    "sparsity_threshold": SPARSITY_THRESHOLD,
    "base_features_kept": len(preprocessor.columns_to_keep_),
    "final_features": X_train.shape[1],
    "tuning_strategy": "one_parameter_at_a_time_sequential",
    "selected_steps": selected_steps,
    "best_params": current_params,
    "test_rmsle": float(root_mean_squared_log_error(y_test_raw, y_pred)),
    "test_rmse": float(root_mean_squared_error(y_test_raw, y_pred)),
    "test_mae": float(mean_absolute_error(y_test_raw, y_pred)),
    "test_r2": float(r2_score(y_test_raw, y_pred)),
}

summary

{'target_transform': 'log1p',
 'primary_metric': 'rmsle',
 'feature_setup': 'sparse_threshold_0.9875_plus_rowwise',
 'sparsity_threshold': 0.9875,
 'base_features_kept': 2482,
 'final_features': 2490,
 'tuning_strategy': 'one_parameter_at_a_time_sequential',
 'selected_steps': [{'parameter': 'num_leaves',
   'selected_value': 31,
   'cv_rmsle_mean': 1.3756589695205137,
   'cv_rmsle_std': 0.022255684385063847},
  {'parameter': 'max_depth',
   'selected_value': 6,
   'cv_rmsle_mean': 1.3739288820873143,
   'cv_rmsle_std': 0.030051257922891258},
  {'parameter': 'min_child_samples',
   'selected_value': 10,
   'cv_rmsle_mean': 1.3692039503048368,
   'cv_rmsle_std': 0.03541561283128999},
  {'parameter': 'learning_rate',
   'selected_value': 0.01,
   'cv_rmsle_mean': 1.3599067633482407,
   'cv_rmsle_std': 0.034480529578614874},
  {'parameter': 'n_estimators',
   'selected_value': 500,
   'cv_rmsle_mean': 1.3599067633482407,
   'cv_rmsle_std': 0.034480529578614874},
  {'parameter': 'subsample

## How To Read The Results

- Compare candidates inside one sweep by `cv_rmsle_mean`.
- Each sweep starts from the best parameters selected by previous sweeps.
- The feature setup is fixed: sparse threshold `0.9875` plus row-wise features.
- If two values are very close, prefer the simpler or more conservative choice.
- The next experiment should use the selected parameters as a starting point for final Optuna tuning.

# 08 LGBM Tuning One Parameter At A Time Report

## Goal

The goal of this notebook was to tune LightGBM in a controlled one-parameter-at-a-time workflow using the selected full sparse-filtered row-wise feature setup.

## Feature Setup

- Sparse threshold from notebook `05`: `0.9875`.
- Base features kept after sparse filtering: `2,482`.
- Row-wise features from notebook `06`: `8`.
- Final feature count: `2,490`.
- Target transform: `log1p(target)`.
- Notebook `07` tested top-k feature-importance subsets, but those subsets had worse held-out test RMSLE than the full `2,490`-feature setup, so they were not used here.

## Tuning Strategy

Each sweep changed one parameter at a time and used the best parameters selected by earlier sweeps. Candidates were compared by CV RMSLE on `log1p(target)`.

| parameter | selected value | CV RMSLE mean | CV RMSLE std |
| --- | ---: | ---: | ---: |
| `num_leaves` | 31 | 1.3757 | 0.0223 |
| `max_depth` | 6 | 1.3739 | 0.0301 |
| `min_child_samples` | 10 | 1.3692 | 0.0354 |
| `learning_rate` | 0.01 | 1.3599 | 0.0345 |
| `n_estimators` | 500 | 1.3599 | 0.0345 |
| `subsample` | 0.8 | 1.3599 | 0.0345 |
| `colsample_bytree` | 1.0 | 1.3585 | 0.0366 |
| `reg_alpha` | 0.5 | 1.3573 | 0.0365 |
| `reg_lambda` | 0.0 | 1.3571 | 0.0365 |

The final selected CV score after the sequential sweeps was `1.3571 +/- 0.0365` RMSLE.

## Selected Parameters

- `learning_rate`: `0.01`
- `n_estimators`: `500`
- `num_leaves`: `31`
- `max_depth`: `6`
- `min_child_samples`: `10`
- `subsample`: `0.8`
- `subsample_freq`: `1`
- `colsample_bytree`: `1.0`
- `reg_alpha`: `0.5`
- `reg_lambda`: `0.0`
- `min_split_gain`: `0.0`

## Final Test Result

| metric | value |
| --- | ---: |
| Test RMSLE | 1.3776 |
| Test RMSE | 7,030,480.10 |
| Test MAE | 3,925,190.76 |
| Test R2 | 0.2256 |

## Conclusion

Manual sequential tuning improved CV RMSLE from notebook `06` (`1.3757`) to `1.3571` and improved held-out test RMSLE from `1.3851` to `1.3776`. Raw-space metrics became weaker than notebook `06` (`RMSE` increased from about `6.89M` to `7.03M`, `MAE` from about `3.86M` to `3.93M`, and `R2` decreased from `0.2570` to `0.2256`), so the gain is mainly on the primary RMSLE objective. The selected parameters are a reasonable starting point for final Optuna tuning on the full sparse-filtered row-wise feature setup.
